# Hugging Face Veri Seti Kalite Analizi

## tl;dr

- Dokuz veri setinde toplam **3.119 satır** incelendi; sekiz sohbet veri setindeki satırların tamamı ve 103 satırlık katalog eksiksiz profillendi.
- Dosya bütünlüğü genel olarak iyi: sohbet setlerinde boş içerik, geçersiz rol veya exact satır tekrarı bulunmadı. Buna rağmen Marvel ve felsefe istemlerinde, biyoloji ve kimlik seti cevaplarında yüksek anlam-tekrar ağırlığı var.
- Yalnız üç veri setinde açık lisans bulunuyor; veri setlerinin tamamı tek split içeriyor.
- MEB verisi yüksek alan değeri taşıyor fakat zaman hassasiyeti ve idari/hukuki risk nedeniyle kaynak bağlı kullanım gerektiriyor. İthaki kataloğu yapısal olarak temiz; fiyat güncelliği ve üst kaynak hakları sınırlayıcı. Kültürel/genel bilgi ile e-ticaret setleri düzenli fakat sentetik doğruluk ve politika iddiaları doğrulanmalı.

## Context & Methods

Dokuz herkese açık Hugging Face veri setinin sabit commit kopyaları 20 Temmuz 2026 tarihinde indirildi. Tam satırlar üzerinden şema, tamamlılık, rol sırası, exact ve normalleştirilmiş tekrar, metin uzunluğu, açık düşünme alanı, kişisel veri biçimleri, zaman hassasiyeti, katalog anahtarları ve fiyat tutarlılığı kontrolleri çalıştırıldı.

### Key Assumptions

- Regex tabanlı kişisel veri taraması, isim ve bağlamsal kimlik gibi yapılandırılmamış kişisel veriyi bütünüyle yakalayamaz.
- Normalleştirilmiş tekrar, noktalama ve büyük/küçük harf farklarını yok sayar; anlamsal benzerliğin tamamını ölçmez.
- Lisans değerlendirmesi veri kartındaki bildirime dayanır; üst kaynak kullanım haklarının hukuki doğrulaması değildir.
- Olgusal doğruluk için tam uzman incelemesi yapılmadı; riskler örnek içerik, kaynak yöntemi ve mevcut provenance alanlarına göre raporlandı.

## Data

In [1]:
import json
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path.cwd()
profiles = json.loads((PROJECT_DIR / "outputs" / "data_quality_profiles.json").read_text(encoding="utf-8"))
manual = json.loads((PROJECT_DIR / "outputs" / "manual_findings.json").read_text(encoding="utf-8"))
overlaps = json.loads((PROJECT_DIR / "outputs" / "cross_dataset_overlap.json").read_text(encoding="utf-8"))

len(profiles), sum(item["profile"]["row_count"] for item in profiles)

(9, 3119)

In [2]:
summary_rows = []
for item in profiles:
    profile = item["profile"]
    scan = profile.get("text_scan", {})
    summary_rows.append({
        "dataset": item["dataset_id"],
        "satır": profile["row_count"],
        "biçim": profile["data_shape"],
        "lisans": item["card"]["license"] or "Belirtilmemiş",
        "exact_tekrar": profile["exact_duplicate_rows"],
        "istem_tekrar_%": round(100 * profile.get("user_prompt_duplicates", {}).get("duplicate_rate", 0), 1),
        "cevap_tekrar_%": round(100 * profile.get("assistant_answer_duplicates", {}).get("duplicate_rate", 0), 1),
        "thinking": profile.get("nonempty_thinking_messages", 0),
        "metin_null": profile.get("string_encoded_null_fields", 0),
        "zaman_eşleşmesi": scan.get("time_sensitive_matches", 0),
    })

summary = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
summary

,dataset,satır,biçim,lisans,exact_tekrar,istem_tekrar_%,cevap_tekrar_%,thinking,metin_null,zaman_eşleşmesi
0,Aysenur44/namaz-vakti-identity-tr,4,conversation,Belirtilmemiş,0,0.0,0.0,0,0,0
1,Egertekin/marvel-domain-dataset,177,conversation,Belirtilmemiş,0,75.7,13.6,0,0,1
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,186,conversation,apache-2.0,0,0.0,0.0,186,0,31
3,aliFurkan123/cultural-questions-dataset,500,conversation,mit,0,0.0,0.0,500,0,4
4,gururaser/ithaki-bilimkurgu-klasikleri,103,catalog,cc-by-nc-4.0,0,0.0,0.0,0,0,0
5,namruni/meb-ogretmen-soru-cevap,450,conversation,Belirtilmemiş,0,0.0,0.0,450,0,297
6,nyzmemre/biyoloji-terimleri-turkce-sa,1093,conversation,Belirtilmemiş,0,0.2,21.6,0,0,0
7,sk75/sahin_identity,77,conversation,Belirtilmemiş,0,0.0,64.9,0,462,0
8,yoitsmeyusuf/felsefe_finetune,529,conversation,Belirtilmemiş,0,76.6,0.0,0,0,21


## Results

### Portföy düzeyinde temel bulgular

Aşağıdaki hücreler, lisans ve veri bölümü eksikliği ile tekrar yoğunluğunu tam profilden hesaplar. Ham büyüklük kalite göstergesi olarak yorumlanmamalıdır.

In [3]:
portfolio = pd.Series({
    "veri_seti": len(summary),
    "toplam_satır": int(summary["satır"].sum()),
    "lisans_belirtilen": int((summary["lisans"] != "Belirtilmemiş").sum()),
    "tek_split": len(summary),
    "thinking_içeren_satır": int(summary["thinking"].sum()),
    "metin_null_hatası": int(summary["metin_null"].sum()),
}, name="değer")
portfolio.to_frame()

,değer
veri_seti,9
toplam_satır,3119
lisans_belirtilen,3
tek_split,9
thinking_içeren_satır,1136
metin_null_hatası,462


### Tekrar ve şema riskleri veri setine göre yoğunlaşıyor

Marvel ve felsefe setlerinde aynı kullanıcı sorusunun çok sayıda farklı cevapla eşleşmesi, rastgele train/test bölmesinde güçlü sızıntı riski oluşturur. Biyoloji veri setinde aynı tanım farklı istemlerle yinelenir. Şahin kimlik setinde ise hem cevap tekrarı hem de string olarak saklanan null değerleri öne çıkar.

In [4]:
summary[["dataset", "satır", "istem_tekrar_%", "cevap_tekrar_%", "thinking", "metin_null"]]

,dataset,satır,istem_tekrar_%,cevap_tekrar_%,thinking,metin_null
0,Aysenur44/namaz-vakti-identity-tr,4,0.0,0.0,0,0
1,Egertekin/marvel-domain-dataset,177,75.7,13.6,0,0
2,Mer1Alii/TR-ECommerce-CustomerSupport-Instruct...,186,0.0,0.0,186,0
3,aliFurkan123/cultural-questions-dataset,500,0.0,0.0,500,0
4,gururaser/ithaki-bilimkurgu-klasikleri,103,0.0,0.0,0,0
5,namruni/meb-ogretmen-soru-cevap,450,0.0,0.0,450,0
6,nyzmemre/biyoloji-terimleri-turkce-sa,1093,0.2,21.6,0,0
7,sk75/sahin_identity,77,0.0,64.9,0,462
8,yoitsmeyusuf/felsefe_finetune,529,76.6,0.0,0,0


### Katalog veri seti anahtar ve fiyat kontrollerini geçiyor

İthaki kataloğunda ISBN, kitap URL'si ve kitap–yazar anahtarı benzersiz; ISBN-13 ve fiyat/indirim hesapları tutarlı. En büyük veri kalitesi konusu, bazı alanların yüksek boşluk oranı ile fiyat ve özetlerin zaman/kaynak metadata eksikliğidir.

In [5]:
catalog = next(item["profile"] for item in profiles if item["profile"]["data_shape"] == "catalog")
pd.DataFrame({
    "kontrol": [
        "satır", "exact tekrar", "ISBN tekrarı", "geçersiz ISBN-13",
        "URL tekrarı", "geçersiz URL", "indirim tutarsızlığı",
        "eksik çevirmen", "eksik kapak tipi", "eksik orijinal ad", "eksik yayın tarihi",
    ],
    "değer": [
        catalog["row_count"], catalog["exact_duplicate_rows"], catalog["duplicate_isbn_rows"],
        catalog["invalid_isbn13_rows"], catalog["duplicate_book_url_rows"], catalog["invalid_url_rows"],
        catalog["discount_inconsistency_rows"], catalog["null_counts"]["cevirmen"],
        catalog["null_counts"]["kapak_tipi"], catalog["null_counts"]["orijinal_adi"],
        catalog["null_counts"]["yayin_tarihi"],
    ],
})

,kontrol,değer
0,satır,103
1,exact tekrar,0
2,ISBN tekrarı,0
3,geçersiz ISBN-13,0
4,URL tekrarı,0
5,geçersiz URL,0
6,indirim tutarsızlığı,0
7,eksik çevirmen,62
8,eksik kapak tipi,96
9,eksik orijinal ad,65


### Veri setleri arası ortak şablon var

NamazAsistan ve Şahin kimlik veri setleri dört kullanıcı istemini paylaşıyor. Bu ortaklık, kimlik seed'leri birlikte kullanılacaksa kaynak şablonun belgelenmesini ve ortak istemlerin aynı değerlendirme grubunda tutulmasını gerektirir.

In [6]:
pd.DataFrame(overlaps)

,left,right,shared_user_prompts,shared_assistant_answers,prompt_examples,answer_examples
0,Aysenur44/namaz-vakti-identity-tr,sk75/sahin_identity,4,0,"[adın ne, ne işe yararsın, sen kimsin]",[]


### Bulguların önem düzeyi ve önerilen düzeltmeler

Aşağıdaki tablo, her veri setinin en önemli veri kalite risklerini ve en küçük etkili düzeltmeyi özetler. Değerlendirme bir puanlama değildir; kullanım amacına göre engelleri görünür kılar.

In [7]:
finding_rows = []
for dataset_id, detail in manual["datasets"].items():
    for finding in detail["findings"]:
        finding_rows.append({
            "dataset": dataset_id,
            "önem": finding["severity"],
            "kanıt": finding["evidence"],
            "risk": finding["risk"],
            "düzeltme": finding["remediation"],
        })
findings = pd.DataFrame(finding_rows)
findings

,dataset,önem,kanıt,risk,düzeltme
0,aliFurkan123/cultural-questions-dataset,high,500 asistan mesajının tamamında thinking alanı...,Sentetik olgusal hatalar ve gerekçe hataları b...,"Bağımsız olgu kontrolü, kaynak/citation alanı ..."
1,aliFurkan123/cultural-questions-dataset,medium,Depo adı kültürel soruları ima ederken veri ka...,Kapsam yanlış anlaşılabilir ve kategori denges...,"Depo adını/kartını eşleştir; konu, zorluk ve k..."
2,Aysenur44/namaz-vakti-identity-tr,critical,Yalnız 4 satır var ve bunlar kimlik/yetenek so...,Alan uzmanlığı kazandırmaz; model yalnız isim ...,"Doğrulanmış ibadet içeriği, konum-zaman bağlam..."
3,Aysenur44/namaz-vakti-identity-tr,high,"Lisans, kaynak, amaç, sınırlama ve örnek kulla...",Yeniden kullanım hakkı ve hedeflenen davranış ...,Tam veri kartı ve açık lisans ekle.
4,Aysenur44/namaz-vakti-identity-tr,medium,Dört kullanıcı isteminin tamamı sk75/sahin_ide...,Şablon tekrarına ve veri setleri arası değerle...,Kaynak şablonu belirt; bölme sırasında ortak i...
5,Egertekin/marvel-domain-dataset,high,177 kullanıcı sorusunun 134'ü normalleştirilmi...,Spider-Man ve birkaç genel soru eğitim sinyali...,Her paragraf için özgül soru üret; karakter/ko...
6,Egertekin/marvel-domain-dataset,high,Kaynağın Wikipedia olduğu yazıyor fakat lisans...,Wikipedia'nın atıf ve share-alike koşullarıyla...,Kaynak URL/revizyonu satır düzeyinde ekle ve u...
7,Egertekin/marvel-domain-dataset,medium,Veri kartı instruction/input/output sütunların...,Kullanıcılar yanlış yükleme kodu veya şema var...,Kartı gerçek şemayla eşleştir ve yükleme örneğ...
8,gururaser/ithaki-bilimkurgu-klasikleri,high,"Veriler yayıncı sitesinden, kapak görselleri i...","Veri seti sahibinin CC BY-NC bildirimi, kaynak...",Üst kaynak kullanım koşullarını doğrula; görse...
9,gururaser/ithaki-bilimkurgu-klasikleri,medium,"Çevirmen 62, kapak tipi 96, orijinal ad 65 ve ...",Alan bazlı analiz ve filtreleme kapsaması deği...,Eksik nedenini belirten durum alanı ekle ve al...


## Takeaways

- **Doğrudan kullanılabilirlik yerine kullanım koşulu belirlemek gerekiyor.** Her veri seti farklı bir göreve yarıyor; lisans, kaynak, zaman hassasiyeti ve split tasarımı giderilmeden ortak bir eğitim havuzuna eklenmemeli.
- **İçsel temizlik yeterli fakat semantik tekrar kritik.** Exact tekrarların sıfır olması, aynı sorunun veya cevabın onlarca kez farklı satırlarda bulunmasını engellemiyor.
- **Yüksek riskli alanlar kaynak bağlı olmalı.** MEB ve e-ticaret içerikleri, güncel politika/mevzuat getiren retrieval katmanı ve açık güven sınırları olmadan kesin yanıt üretiminde kullanılmamalı.
- **Katalog verisi diğerlerinden farklı ele alınmalı.** İthaki seti sohbet değil yapılandırılmış ürün kataloğu; fiyat snapshot'ı ve üst kaynak hakları tanımlanırsa analiz ve kontrollü üretim için değerlidir.
- **Kimlik setleri ana eğitim verisi değildir.** Dört ve 77 satırlık kimlik örnekleri düşük ağırlıklı davranış tohumu olarak tutulmalı; alan yetkinliği veya genel kalite kanıtı sayılmamalı.